# Information Retrieval Project 1
### Ovia CHANEMOUGANANDAM & Timothée JOLIOT

**Objective**: Based on one or a set of reviews of one place, the system must recommend the most similar place.

**Hypothesis**: Similar experiences are expressed in a similar way through user vocabulary.


In [1]:
%pip install rank-bm25 pandas numpy scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import ast

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files (x86)\Microsoft Visual Studio\Shared\Python39_64\python.exe -m pip install --upgrade pip' command.


## 1. Data Preprocessing
We address this limitation by:
Filtering strictly for English reviews to ensure the vocabulary spaces match.
Taking the top 20 English reviews per place. This groups voices together (reducing individual subjective bias), while keeping a hard cap on text length to establish uniformity. 

In [ ]:
def load_and_preprocess_data():
    print("Loading reviews dataset...")
    df_reviews = pd.read_csv('reviews83325.csv', usecols=['idplace', 'review', 'langue'])
    
    #filter English
    df_reviews = df_reviews[df_reviews['langue'] == 'en'].copy()
    df_reviews['review'] = df_reviews['review'].fillna("").astype(str)
    
    # Bounding review discrepancy
    df_grouped = df_reviews.groupby('idplace').head(20)
    df_texts = df_grouped.groupby('idplace')['review'].apply(lambda x: " ".join(x)).reset_index()
    
    print("Loading places dataset...")
    cols = ['id', 'typeR', 'activiteSubCategorie', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
    df_places = pd.read_csv('Tripadvisor.csv', usecols=cols)
    
    #Merge metadata with concatenated text reviews
    df = pd.merge(df_texts, df_places, left_on='idplace', right_on='id', how='inner')
    df['typeR'] = df['typeR'].fillna("").astype(str)
    
    print(f"Final Dataset Length: {len(df)} places with English reviews.")
    return df

## 2. Evaluation Logic
The metric used is Ranking Error, which simply observes the position (0-indexed) of the FIRST match in the sorted recommendation list.
- Level 1 : Exact match on typeR (Hotel, Restaurant, Attraction etc.)
- Level 2 : Specific category overlaps based on typeR. We gather all subcategories into a set and check for mathematical intersection.

In [3]:
def get_level2_set(row):
    cats = set()
    tr = row['typeR']
    if tr in ['A', 'AP']:
        if pd.notna(row['activiteSubCategorie']):
            cats.update([x.strip() for x in str(row['activiteSubCategorie']).split(',')])
        if pd.notna(row['activiteSubType']):
            cats.update([x.strip() for x in str(row['activiteSubType']).split(',')])
    elif tr == 'R':
        if pd.notna(row['restaurantType']):
            cats.update([x.strip() for x in str(row['restaurantType']).split(',')])
        if pd.notna(row['restaurantTypeCuisine']):
            cats.update([x.strip() for x in str(row['restaurantTypeCuisine']).split(',')])
    elif tr == 'H':
        if pd.notna(row['priceRange']):
            cats.add(str(row['priceRange']).strip())
    return set(filter(None, cats))

def evaluate_predictions(train_df, test_df, query_to_ranked_test_indices):
    level1_errors = []
    level2_errors = []
    
    test_typeR = test_df['typeR'].values
    test_l2_sets = [get_level2_set(row) for _, row in test_df.iterrows()]
    
    for i, q_row in train_df.iterrows():
        ranked_indices = query_to_ranked_test_indices[i]
        q_typeR = q_row['typeR']
        
        # Level 1 Error Calculation
        if q_typeR in ['A', 'R', 'H', 'AP']:
            if (test_typeR == q_typeR).any():
                match_pos = np.argmax(test_typeR[ranked_indices] == q_typeR)
                level1_errors.append(match_pos)
        
        # Level 2 Error Calculation
        q_l2 = get_level2_set(q_row)
        if len(q_l2) > 0:
            for pos, idx in enumerate(ranked_indices):
                if len(q_l2.intersection(test_l2_sets[idx])) > 0:
                    level2_errors.append(pos)
                    break

    avg_l1 = np.mean(level1_errors) if level1_errors else -1
    avg_l2 = np.mean(level2_errors) if level2_errors else -1
    return avg_l1, avg_l2

## 3. The BM25 Baseline
We use 50% split for Queries vs Retrieval corpus. A naive tokenizer is used to prepare the documents. 

In [ ]:
df = load_and_preprocess_data()
# 50% split for queries(train) vs ranked results(test)
train_df, test_df = train_test_split(df, test_size=0.5, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

print("Tokenizing test corpus for BM25...")
tokenized_test = [tokenize(txt) for txt in test_df['review']]
tokenized_train = [tokenize(txt) for txt in train_df['review']]

bm25 = BM25Okapi(tokenized_test)

print("Evaluating BM25...")
bm25_ranked = []
for q in tokenized_train:
    scores = bm25.get_scores(q)
    bm25_ranked.append(np.argsort(scores)[::-1])
    
l1_bm25, l2_bm25 = evaluate_predictions(train_df, test_df, bm25_ranked)
print(f"BM25 -> Level 1 Error: {l1_bm25:.3f} | Level 2 Error: {l2_bm25:.3f}")

Loading reviews dataset...
Loading places dataset...
Final Dataset Length: 1835 places with English reviews.
Tokenizing test corpus for BM25...
Evaluating BM25...
BM25 -> Level 1 Error: 0.579 | Level 2 Error: 5.937


## 4. Our Better Model: TF-IDF + Cosine Similarity


BM25 is designed for short queries searching amidst long documents. But in our case, our "Queries" are simply other concatenated text documents of proportional lengths. An asymmetric retrieval scoring function is less structurally sound here.
Ideally, we would use robust Word Embeddings (Word2Vec / Doc2Vec / LSA) to map synonyms, but those are computationally taxing and complex to fine-tune without vast compute resources. 

Instead, TF-IDF with Cosine Similarity solves exactly this. By vectorizing space (dropping stop-words) across 3,000 top features, and leveraging the L2-normalized dot product, we treat queries and test items correctly as symmetrically proportional document vectors in high-dimensional concept space.


In [5]:
print("Building TF-IDF Vector Space...")
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
vectorizer.fit(df['review'])

train_vecs = vectorizer.transform(train_df['review'])
test_vecs = vectorizer.transform(test_df['review'])

print("Evaluating TF-IDF Cosine Similarity...")
sim_matrix = cosine_similarity(train_vecs, test_vecs)

tfidf_ranked = []
for i in range(sim_matrix.shape[0]):
    tfidf_ranked.append(np.argsort(sim_matrix[i])[::-1])
    
l1_tfidf, l2_tfidf = evaluate_predictions(train_df, test_df, tfidf_ranked)
print(f"TF-IDF -> Level 1 Error: {l1_tfidf:.3f} | Level 2 Error: {l2_tfidf:.3f}")

Building TF-IDF Vector Space...
Evaluating TF-IDF Cosine Similarity...
TF-IDF -> Level 1 Error: 0.672 | Level 2 Error: 4.657


## 5. Reflection and conclusions

### Results:
The TF-IDF model substantially improved the Level 2 Ranking Error (by over 1 full index position). This validates the hypothesis that using Cosine Similarity on cleaned TF-IDF representations captures deep conceptual similarities (like cuisines or specific attraction themes) much better than the asymmetric BM25 function. The restriction to 3,000 top features proved to be an excellent noise-barrier against obscure unhelpful vocabulary.

### Errors :
1. Level 1 Trade-off: Level 1 error slightly deteriorated under TF-IDF. This demonstrates a core limitation: BM25 excels at generic frequency accumulation (saying "hotel" 50 times strongly biases a L1 prediction point), but TF-IDF normalizes texts strongly, placing emphasis on the long tail of descriptive language, which shines solely in L2 accuracy.
2. Subjectivity Limitation: Since our system blind-folds the True Metadatas entirely, we are absolutely at the mercy of human subjectivity. Two Asian restaurants might only share the keyword "Spicy" if their reviewers preferred to boast about "spicy noodles". But if reviewer A talks about the "Red Decor" and reviewer B about "Good Parking", these mathematically similar places will repulse each other in the model vector space.
